In [0]:
# Test — Day 7 Performance Optimization & Churn Metrics

import unittest

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("gold_schema", "gold", "2. Gold Schema")
CATALOG = dbutils.widgets.get("catalog_name")
GOLD = dbutils.widgets.get("gold_schema")


class PerformanceDay7Tests(unittest.TestCase):

    def test_liquid_and_partitioned_tables_match_source_row_count(self):
        source = spark.table(f"{CATALOG}.{GOLD}.fact_street_readings").count()
        liquid = spark.table(f"{CATALOG}.{GOLD}.fact_street_readings_liquid").count()
        # partitioned table includes MERGE demo rows (+10) if 17 already ran; liquid does too
        self.assertGreaterEqual(liquid, source, "Liquid table has fewer rows than source.")

    def test_benchmark_results_has_5_queries(self):
        df = spark.table(f"{CATALOG}.{GOLD}.benchmark_results")
        self.assertEqual(df.count(), 5, f"Expected 5 benchmark rows, got {df.count()}.")

    def test_benchmark_winner_is_liquid_or_zorder(self):
        winners = {r["winner"] for r in spark.table(f"{CATALOG}.{GOLD}.benchmark_results").select("winner").collect()}
        self.assertTrue(winners.issubset({"Liquid", "Z-Order"}), f"Unexpected winner values: {winners}")

    def test_merge_demo_rows_present_and_exactly_10(self):
        """
        FIXED: fact_street_readings has no reading_id column (only
        fact_traffic_counts does, since only cars.csv has an id field).
        Marker is a synthetic future reading_ts instead — real data ends
        2024-03-11, so anything >= 2025-01-01 is unambiguously synthetic.
        """
        df = spark.table(f"{CATALOG}.{GOLD}.fact_street_readings_liquid").filter("reading_ts >= '2025-01-01'")
        self.assertEqual(df.count(), 10, f"Expected exactly 10 synthetic late-arriving rows, got {df.count()}.")

    def test_agg_stale_streets_covers_all_36_streets(self):
        """Confirms the LEFT JOIN from dim_street worked — never-reported streets must still appear."""
        df = spark.table(f"{CATALOG}.{GOLD}.agg_stale_streets")
        self.assertEqual(df.count(), 36, f"Expected 36 streets scored, got {df.count()}.")

    def test_agg_stale_streets_status_values_are_valid(self):
        valid = {"ACTIVE", "AT_RISK", "STALE", "CRITICAL", "NEVER_REPORTED"}
        statuses = {r["churn_status"] for r in
                    spark.table(f"{CATALOG}.{GOLD}.agg_stale_streets").select("churn_status").collect()}
        self.assertTrue(statuses.issubset(valid), f"Unexpected churn_status values: {statuses - valid}")

    def test_agg_danger_tier_churn_pct_is_between_0_and_100(self):
        df = spark.table(f"{CATALOG}.{GOLD}.agg_danger_tier_churn")
        out_of_range = df.filter("churn_score_pct < 0 OR churn_score_pct > 100").count()
        self.assertEqual(out_of_range, 0, f"{out_of_range} tiers have an out-of-range churn_score_pct.")

    def test_liquid_table_uses_cluster_by_not_partitioned(self):
        props = dict(spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{GOLD}.fact_street_readings_liquid").collect())
        self.assertEqual(props.get("optimization"), "liquid_clustering")

    def test_partitioned_table_uses_zorder(self):
        props = dict(spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{GOLD}.fact_street_readings_partitioned").collect())
        self.assertEqual(props.get("optimization"), "partition_zorder")


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(PerformanceDay7Tests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 7 tests FAILED — see output above.")
